# baseline_refresh

Rebuilds `AnomalyBaseline` in `ci-eventhouse` from the last 4 weeks of `CapacitySummary`: mean and standard deviation of CU % per capacity and hour of week. The Summary update policy scores every new row against this table, so a fresher baseline means better anomaly scores.

Runs on a Fabric Python notebook (no Spark session), scheduled hourly. It makes one management call to the Eventhouse and changes nothing else.

In [ ]:
# Parameters (override from the schedule or a pipeline if needed)
CLUSTER = "https://trd-3p56kn96phc55ysd7n.z5.kusto.fabric.microsoft.com"
DATABASE = "ci-eventhouse"
LOOKBACK_DAYS = 28

In [ ]:
import requests

# Token for the Eventhouse query endpoint, issued for the identity that runs the notebook.
token = notebookutils.credentials.getToken(CLUSTER)

csl = f"""
.set-or-replace AnomalyBaseline <|
CapacitySummary
| where Timestamp > ago({LOOKBACK_DAYS}d) and isnotnull(CuPercent)
| extend HourOfWeek = toint(dayofweek(Timestamp) / 1d) * 24 + hourofday(Timestamp)
| summarize Mean = toreal(avg(CuPercent)), Std = toreal(stdev(CuPercent)), Samples = count() by CapacityId, HourOfWeek
| extend UpdatedAt = now()
| project CapacityId, HourOfWeek = toint(HourOfWeek), Mean, Std, Samples, UpdatedAt
"""

resp = requests.post(
    f"{CLUSTER}/v1/rest/mgmt",
    json={"db": DATABASE, "csl": csl},
    headers={"Authorization": f"Bearer {token}"},
    timeout=120,
)
resp.raise_for_status()
rows = resp.json()["Tables"][0]["Rows"]
print(f"AnomalyBaseline rebuilt: {sum(r[-1] for r in rows)} rows in {len(rows)} extent(s)")